In [1]:
# ============================================
# ONE-CELL DBLOSS PILOT
# Run this in a NEW notebook: 07_dbloss_pilot.ipynb
# ============================================

from pathlib import Path
import sys
import importlib
import gc
import json
import math
import random
import copy

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F

# --------------------------------------------
# 0) Project setup
# --------------------------------------------
ROOT = Path("/data/Sajjan_Singh/spml/wavelet_seq_project").resolve()
SCRIPTS_DIR = ROOT / "scripts"
RUNNER_PATH = SCRIPTS_DIR / "phase5_adaptive_wavelet_model.py"

if not RUNNER_PATH.exists():
    raise FileNotFoundError(f"Missing runner file: {RUNNER_PATH}")

if str(SCRIPTS_DIR) not in sys.path:
    sys.path.append(str(SCRIPTS_DIR))

import phase5_adaptive_wavelet_model as p5
importlib.reload(p5)

OUT_TABLES = ROOT / "results" / "tables" / "phase5"
OUT_PREDS = ROOT / "results" / "predictions" / "phase5_dbloss_pilot"
OUT_CKPTS = ROOT / "results" / "checkpoints" / "phase5_dbloss_pilot"

OUT_TABLES.mkdir(parents=True, exist_ok=True)
OUT_PREDS.mkdir(parents=True, exist_ok=True)
OUT_CKPTS.mkdir(parents=True, exist_ok=True)

MASTER_BEST_PATH = ROOT / "results" / "tables" / "master_best_by_dataset_unified.csv"

print("Imported p5 from:", RUNNER_PATH)
print("DEVICE:", p5.DEVICE)

# --------------------------------------------
# 1) Fixed pilot settings
# --------------------------------------------
PILOT_DATASETS = ["etth1", "weather", "traffic", "ili"]

BEST_ARCH = {
    "levels": 2,
    "wavelet_family": "Db4",
    "d_model": 128,
    "num_backbone_blocks": 2,
    "dropout": 0.10,
    "filter_reg_lambda": 1e-4,
    "gate_entropy_lambda": 1e-4,
}

# This is a practical decomposition-aware pilot loss inspired by the idea
# of supervising trend + residual/seasonal components separately.
# It is NOT claimed here as the official paper reproduction.
DBLOSS_CFG = {
    "lambda_decomp": 0.35,
    "trend_weight": 1.0,
    "seasonal_weight": 1.0,
    # odd kernel sizes for moving-average trend extraction
    "kernel_map": {
        "etth1": 25,
        "weather": 49,
        "traffic": 25,
        "ili": 7,
    },
}

SEED = 42
USE_AMP = bool(torch.cuda.is_available()) and bool(getattr(p5, "TRAIN_CFG", {}).get("use_amp", True))
PATIENCE = int(getattr(p5, "TRAIN_CFG", {}).get("full_patience", 8))

print("PILOT_DATASETS:", PILOT_DATASETS)
print("BEST_ARCH:", BEST_ARCH)
print("DBLOSS_CFG:", DBLOSS_CFG)
print("USE_AMP:", USE_AMP)
print("PATIENCE:", PATIENCE)

# --------------------------------------------
# 2) Helpers
# --------------------------------------------
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

def mse(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=np.float64)
    y_pred = np.asarray(y_pred, dtype=np.float64)
    return float(np.mean((y_true - y_pred) ** 2))

def mae(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=np.float64)
    y_pred = np.asarray(y_pred, dtype=np.float64)
    return float(np.mean(np.abs(y_true - y_pred)))

def rmse(y_true, y_pred):
    return float(np.sqrt(mse(y_true, y_pred)))

def resolve_seq_len(ds):
    seq_candidates = p5.resolve_seq_candidates(ds, "searched")
    return int(seq_candidates[min(1, len(seq_candidates) - 1)])

def resolve_batch_size(ds):
    return int(p5.DEFAULT_BATCH_SIZE[ds])

def resolve_epochs(ds):
    return int(p5.DEFAULT_EPOCHS[ds])

def safe_kernel(kernel, pred_len):
    k = int(kernel)
    if k >= pred_len:
        k = pred_len - 1 if pred_len % 2 == 0 else pred_len
    if k < 3:
        k = 3 if pred_len >= 3 else 1
    if k % 2 == 0:
        k -= 1
    if k < 1:
        k = 1
    return k

def moving_average_trend(x, kernel):
    # x: [B, H]
    if kernel <= 1:
        return x
    pad = kernel // 2
    x3 = x.unsqueeze(1)  # [B,1,H]
    xpad = F.pad(x3, (pad, pad), mode="replicate")
    trend = F.avg_pool1d(xpad, kernel_size=kernel, stride=1)
    return trend.squeeze(1)

def decomposition_aware_loss(pred_scaled, y_scaled, dataset_name, cfg):
    base = F.mse_loss(pred_scaled, y_scaled)

    kernel = safe_kernel(cfg["kernel_map"][dataset_name], pred_scaled.shape[1])

    trend_pred = moving_average_trend(pred_scaled, kernel)
    trend_true = moving_average_trend(y_scaled, kernel)

    seasonal_pred = pred_scaled - trend_pred
    seasonal_true = y_scaled - trend_true

    trend_loss = F.mse_loss(trend_pred, trend_true)
    seasonal_loss = F.mse_loss(seasonal_pred, seasonal_true)

    total = (
        base
        + cfg["lambda_decomp"] * (
            cfg["trend_weight"] * trend_loss
            + cfg["seasonal_weight"] * seasonal_loss
        )
    )

    parts = {
        "base_mse": float(base.detach().cpu().item()),
        "trend_loss": float(trend_loss.detach().cpu().item()),
        "seasonal_loss": float(seasonal_loss.detach().cpu().item()),
        "kernel": kernel,
    }
    return total, parts

@torch.no_grad()
def evaluate_model(model, loader, target_mean, target_std):
    model.eval()
    preds = []
    trues = []

    for batch in loader:
        x = batch["x_scaled"].to(p5.DEVICE, non_blocking=True)
        y_raw = batch["y_raw"].cpu().numpy()

        with torch.cuda.amp.autocast(enabled=USE_AMP):
            pred_scaled, aux = model(x)

        pred_raw = pred_scaled.detach().cpu().numpy() * target_std + target_mean
        preds.append(pred_raw)
        trues.append(y_raw)

    preds = np.concatenate(preds, axis=0)
    trues = np.concatenate(trues, axis=0)

    return {
        "preds": preds,
        "trues": trues,
        "mse": mse(trues.reshape(-1), preds.reshape(-1)),
        "mae": mae(trues.reshape(-1), preds.reshape(-1)),
        "rmse": rmse(trues.reshape(-1), preds.reshape(-1)),
    }

def train_one_run(dataset_name, variant_name, use_dbloss):
    set_seed(SEED)

    bundle = p5.load_bundle(dataset_name, input_mode="multivariate")
    pred_len = p5.resolve_pred_len(dataset_name, "long")
    seq_len = resolve_seq_len(dataset_name)
    batch_size = resolve_batch_size(dataset_name)
    epochs = resolve_epochs(dataset_name)

    train_loader, val_loader, test_loader = p5.make_loaders(bundle, seq_len, pred_len, batch_size)

    target_idx = bundle["target_idx"]
    target_mean = float(bundle["scaler_mean"][target_idx])
    target_std = float(bundle["scaler_std"][target_idx])

    model = p5.AdaptiveWaveletMixer(
        input_dim=int(bundle["values_scaled"].shape[1]),
        pred_len=pred_len,
        d_model=int(BEST_ARCH["d_model"]),
        levels=int(BEST_ARCH["levels"]),
        wavelet_family=BEST_ARCH["wavelet_family"],
        num_backbone_blocks=int(BEST_ARCH["num_backbone_blocks"]),
        dropout=float(BEST_ARCH["dropout"]),
        filter_reg_lambda=float(BEST_ARCH["filter_reg_lambda"]),
        gate_entropy_lambda=float(BEST_ARCH["gate_entropy_lambda"]),
    ).to(p5.DEVICE)

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=1e-3,
        weight_decay=1e-5,
    )

    scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP)

    best_val_rmse = float("inf")
    best_epoch = -1
    best_state = None

    history = []

    for epoch in range(1, epochs + 1):
        model.train()
        train_losses = []

        for batch in train_loader:
            x = batch["x_scaled"].to(p5.DEVICE, non_blocking=True)
            y = batch["y_scaled"].to(p5.DEVICE, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)

            with torch.cuda.amp.autocast(enabled=USE_AMP):
                pred_scaled, aux = model(x)

                if use_dbloss:
                    loss, loss_parts = decomposition_aware_loss(pred_scaled, y, dataset_name, DBLOSS_CFG)
                else:
                    loss = F.mse_loss(pred_scaled, y)
                    loss_parts = {
                        "base_mse": float(loss.detach().cpu().item()),
                        "trend_loss": np.nan,
                        "seasonal_loss": np.nan,
                        "kernel": np.nan,
                    }

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            train_losses.append(float(loss.detach().cpu().item()))

        val_eval = evaluate_model(model, val_loader, target_mean, target_std)
        val_rmse = float(val_eval["rmse"])
        mean_train_loss = float(np.mean(train_losses)) if len(train_losses) > 0 else np.nan

        history.append({
            "dataset": dataset_name,
            "variant": variant_name,
            "epoch": epoch,
            "train_loss": mean_train_loss,
            "val_rmse": val_rmse,
            "val_mae": float(val_eval["mae"]),
            "val_mse": float(val_eval["mse"]),
            "base_mse_component": loss_parts["base_mse"],
            "trend_loss_component": loss_parts["trend_loss"],
            "seasonal_loss_component": loss_parts["seasonal_loss"],
            "decomp_kernel": loss_parts["kernel"],
        })

        print(
            f"[{dataset_name} | {variant_name}] "
            f"epoch {epoch:03d} | train_loss={mean_train_loss:.6f} | val_rmse={val_rmse:.6f}"
        )

        if val_rmse < best_val_rmse:
            best_val_rmse = val_rmse
            best_epoch = epoch
            best_state = copy.deepcopy(model.state_dict())
            patience_left = PATIENCE
        else:
            patience_left -= 1
            if patience_left <= 0:
                print(f"[{dataset_name} | {variant_name}] early stopping at epoch {epoch}")
                break

    model.load_state_dict(best_state)

    test_eval = evaluate_model(model, test_loader, target_mean, target_std)

    ckpt_path = OUT_CKPTS / f"{dataset_name}_{variant_name}.pt"
    pred_path = OUT_PREDS / f"{dataset_name}_{variant_name}_test_predictions.npz"

    torch.save({
        "dataset": dataset_name,
        "variant": variant_name,
        "best_arch": BEST_ARCH,
        "use_dbloss": use_dbloss,
        "dbloss_cfg": DBLOSS_CFG if use_dbloss else None,
        "seq_len": seq_len,
        "pred_len": pred_len,
        "best_epoch": best_epoch,
        "state_dict": model.state_dict(),
    }, ckpt_path)

    np.savez_compressed(
        pred_path,
        preds=test_eval["preds"],
        trues=test_eval["trues"],
        seq_len=seq_len,
        pred_len=pred_len,
    )

    row = {
        "dataset": dataset_name,
        "variant": variant_name,
        "model": "AdaptiveWaveletMixer_bestcfg" if not use_dbloss else "AdaptiveWaveletMixer_bestcfg_DBLossPilot",
        "family": "wavelet_adaptive",
        "use_dbloss": bool(use_dbloss),
        "seq_len": seq_len,
        "pred_len": pred_len,
        "best_epoch": int(best_epoch),
        "checkpoint": str(ckpt_path.relative_to(ROOT)),
        "prediction_file": str(pred_path.relative_to(ROOT)),
        "test_mse": float(test_eval["mse"]),
        "test_mae": float(test_eval["mae"]),
        "test_rmse": float(test_eval["rmse"]),
    }

    hist_df = pd.DataFrame(history)

    del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return row, hist_df

# --------------------------------------------
# 3) Run pilot: baseline vs DBLoss pilot
# --------------------------------------------
all_rows = []
all_hist = []

EXPERIMENTS = [
    ("baseline", False),
    ("dbloss_pilot", True),
]

print("\nStarting 4-dataset pilot...")
for ds in PILOT_DATASETS:
    for variant_name, use_dbloss in EXPERIMENTS:
        print("\n" + "=" * 120)
        print(f"RUNNING | dataset={ds} | variant={variant_name}")
        print("=" * 120)
        row, hist_df = train_one_run(ds, variant_name, use_dbloss)
        all_rows.append(row)
        all_hist.append(hist_df)

pilot_df = pd.DataFrame(all_rows)
hist_df = pd.concat(all_hist, ignore_index=True)

pilot_csv = OUT_TABLES / "phase5_dbloss_pilot_metrics.csv"
hist_csv = OUT_TABLES / "phase5_dbloss_pilot_history.csv"

pilot_df.to_csv(pilot_csv, index=False)
hist_df.to_csv(hist_csv, index=False)

print("\nSaved pilot metrics:", pilot_csv)
print("Saved pilot history:", hist_csv)
display(pilot_df)

# --------------------------------------------
# 4) Summary comparison
# --------------------------------------------
pivot_rmse = pilot_df.pivot(index="dataset", columns="variant", values="test_rmse").reset_index()
pivot_mae = pilot_df.pivot(index="dataset", columns="variant", values="test_mae").reset_index()

summary = pivot_rmse.merge(pivot_mae, on="dataset", suffixes=("_rmse", "_mae"))

summary["rmse_gain_abs"] = summary["baseline_rmse"] - summary["dbloss_pilot_rmse"]
summary["rmse_gain_pct"] = 100.0 * summary["rmse_gain_abs"] / summary["baseline_rmse"]

summary["mae_gain_abs"] = summary["baseline_mae"] - summary["dbloss_pilot_mae"]
summary["mae_gain_pct"] = 100.0 * summary["mae_gain_abs"] / summary["baseline_mae"]

if MASTER_BEST_PATH.exists():
    master_best = pd.read_csv(MASTER_BEST_PATH)
    master_sub = master_best[master_best["dataset"].isin(PILOT_DATASETS)].copy()
    master_sub = master_sub.rename(columns={
        "model": "current_best_model",
        "family": "current_best_family",
        "test_rmse": "current_best_rmse",
        "test_mae": "current_best_mae",
        "test_mse": "current_best_mse",
    })
    summary = summary.merge(master_sub, on="dataset", how="left")

summary_csv = OUT_TABLES / "phase5_dbloss_pilot_summary.csv"
summary.to_csv(summary_csv, index=False)

print("\nSaved summary:", summary_csv)
display(summary)

# --------------------------------------------
# 5) Automatic recommendation
# --------------------------------------------
wins = int((summary["rmse_gain_abs"] > 0).sum())
losses = int((summary["rmse_gain_abs"] < 0).sum())
ties = int((summary["rmse_gain_abs"] == 0).sum())

print("\n" + "=" * 120)
print("PILOT DECISION")
print("=" * 120)
print(f"DBLoss pilot wins on {wins}/{len(PILOT_DATASETS)} datasets by RMSE")
print(f"DBLoss pilot loses on {losses}/{len(PILOT_DATASETS)} datasets by RMSE")
print(f"Ties: {ties}")

if wins >= 3:
    print("RECOMMENDATION: DBLoss pilot looks promising. Run on all 9 datasets next.")
else:
    print("RECOMMENDATION: DBLoss pilot is not strong enough yet. Do NOT scale to all 9 immediately.")

print("\nThis pilot uses a practical decomposition-aware loss inspired by DBLoss,")
print("not a claim of exact official reproduction.")

Imported p5 from: /data/Sajjan_Singh/spml/wavelet_seq_project/scripts/phase5_adaptive_wavelet_model.py
DEVICE: cuda
PILOT_DATASETS: ['etth1', 'weather', 'traffic', 'ili']
BEST_ARCH: {'levels': 2, 'wavelet_family': 'Db4', 'd_model': 128, 'num_backbone_blocks': 2, 'dropout': 0.1, 'filter_reg_lambda': 0.0001, 'gate_entropy_lambda': 0.0001}
DBLOSS_CFG: {'lambda_decomp': 0.35, 'trend_weight': 1.0, 'seasonal_weight': 1.0, 'kernel_map': {'etth1': 25, 'weather': 49, 'traffic': 25, 'ili': 7}}
USE_AMP: True
PATIENCE: 5

Starting 4-dataset pilot...

RUNNING | dataset=etth1 | variant=baseline


/tmp/ipykernel_3015075/2559884152.py:235: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP)
/tmp/ipykernel_3015075/2559884152.py:253: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=USE_AMP):


[etth1 | baseline] epoch 001 | train_loss=0.249523 | val_rmse=4.138276


/tmp/ipykernel_3015075/2559884152.py:184: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=USE_AMP):


[etth1 | baseline] epoch 002 | train_loss=0.158984 | val_rmse=4.340668
[etth1 | baseline] epoch 003 | train_loss=0.153027 | val_rmse=5.316027
[etth1 | baseline] epoch 004 | train_loss=0.148266 | val_rmse=5.524295
[etth1 | baseline] epoch 005 | train_loss=0.134957 | val_rmse=6.397129
[etth1 | baseline] epoch 006 | train_loss=0.153661 | val_rmse=5.195210
[etth1 | baseline] early stopping at epoch 6

RUNNING | dataset=etth1 | variant=dbloss_pilot
[etth1 | dbloss_pilot] epoch 001 | train_loss=0.333176 | val_rmse=4.132932
[etth1 | dbloss_pilot] epoch 002 | train_loss=0.210954 | val_rmse=4.348708
[etth1 | dbloss_pilot] epoch 003 | train_loss=0.203165 | val_rmse=5.296891
[etth1 | dbloss_pilot] epoch 004 | train_loss=0.196741 | val_rmse=5.500023
[etth1 | dbloss_pilot] epoch 005 | train_loss=0.179029 | val_rmse=6.365869
[etth1 | dbloss_pilot] epoch 006 | train_loss=0.202980 | val_rmse=5.195192
[etth1 | dbloss_pilot] early stopping at epoch 6

RUNNING | dataset=weather | variant=baseline
[weathe

,dataset,variant,model,family,use_dbloss,seq_len,pred_len,best_epoch,checkpoint,prediction_file,test_mse,test_mae,test_rmse
0,etth1,baseline,AdaptiveWaveletMixer_bestcfg,wavelet_adaptive,False,96,96,1,results/checkpoints/phase5_dbloss_pilot/etth1_...,results/predictions/phase5_dbloss_pilot/etth1_...,2.280040e+01,4.251028,4.774976
1,etth1,dbloss_pilot,AdaptiveWaveletMixer_bestcfg_DBLossPilot,wavelet_adaptive,True,96,96,1,results/checkpoints/phase5_dbloss_pilot/etth1_...,results/predictions/phase5_dbloss_pilot/etth1_...,2.270246e+01,4.239979,4.764709
2,weather,baseline,AdaptiveWaveletMixer_bestcfg,wavelet_adaptive,False,192,96,2,results/checkpoints/phase5_dbloss_pilot/weathe...,results/predictions/phase5_dbloss_pilot/weathe...,4.687624e+02,15.991391,21.650922
3,weather,dbloss_pilot,AdaptiveWaveletMixer_bestcfg_DBLossPilot,wavelet_adaptive,True,192,96,6,results/checkpoints/phase5_dbloss_pilot/weathe...,results/predictions/phase5_dbloss_pilot/weathe...,4.546590e+02,13.142364,21.322735
4,traffic,baseline,AdaptiveWaveletMixer_bestcfg,wavelet_adaptive,False,168,96,16,results/checkpoints/phase5_dbloss_pilot/traffi...,results/predictions/phase5_dbloss_pilot/traffi...,9.388724e-05,0.006606,0.009690
5,traffic,dbloss_pilot,AdaptiveWaveletMixer_bestcfg_DBLossPilot,wavelet_adaptive,True,168,96,17,results/checkpoints/phase5_dbloss_pilot/traffi...,results/predictions/phase5_dbloss_pilot/traffi...,9.075994e-05,0.006540,0.009527
6,ili,baseline,AdaptiveWaveletMixer_bestcfg,wavelet_adaptive,False,52,24,21,results/checkpoints/phase5_dbloss_pilot/ili_ba...,results/predictions/phase5_dbloss_pilot/ili_ba...,1.931418e+11,382121.493091,439479.031688
7,ili,dbloss_pilot,AdaptiveWaveletMixer_bestcfg_DBLossPilot,wavelet_adaptive,True,52,24,21,results/checkpoints/phase5_dbloss_pilot/ili_db...,results/predictions/phase5_dbloss_pilot/ili_db...,1.934174e+11,382260.229412,439792.473989



Saved summary: /data/Sajjan_Singh/spml/wavelet_seq_project/results/tables/phase5/phase5_dbloss_pilot_summary.csv


,dataset,baseline_rmse,dbloss_pilot_rmse,baseline_mae,dbloss_pilot_mae,rmse_gain_abs,rmse_gain_pct,mae_gain_abs,mae_gain_pct,current_best_model,current_best_family,current_best_mse,current_best_mae,current_best_rmse
0,etth1,4.774976,4.764709,4.251028,4.239979,0.010267,0.215014,0.011050,0.259929,WPMixer,modern_tslib,4.691051e+00,1.635539,2.165884
1,ili,439479.031688,439792.473989,382121.493091,382260.229412,-313.442302,-0.071321,-138.736320,-0.036307,iTransformer,modern_tslib,3.287862e+10,125665.623324,181324.636078
2,traffic,0.009690,0.009527,0.006606,0.006540,0.000163,1.679560,0.000066,0.992387,dlinear,neural_non_wavelet,4.471525e-05,0.003913,0.006687
3,weather,21.650922,21.322735,15.991391,13.142364,0.328187,1.515810,2.849028,17.816008,WPMixer,modern_tslib,1.728212e+02,10.047792,13.146148



PILOT DECISION
DBLoss pilot wins on 3/4 datasets by RMSE
DBLoss pilot loses on 1/4 datasets by RMSE
Ties: 0
RECOMMENDATION: DBLoss pilot looks promising. Run on all 9 datasets next.

This pilot uses a practical decomposition-aware loss inspired by DBLoss,
not a claim of exact official reproduction.


In [ ]:
# ============================================
# ONE-CELL DBLOSS FULL RUN ON ALL 9 DATASETS
# Run this in a NEW notebook: 08_dbloss_fullrun.ipynb
# ============================================

from pathlib import Path
import sys
import importlib
import gc
import json
import math
import random
import copy
import time

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F

# --------------------------------------------
# 0) Project setup
# --------------------------------------------
ROOT = Path("/data/Sajjan_Singh/spml/wavelet_seq_project").resolve()
SCRIPTS_DIR = ROOT / "scripts"
RUNNER_PATH = SCRIPTS_DIR / "phase5_adaptive_wavelet_model.py"

if not RUNNER_PATH.exists():
    raise FileNotFoundError(f"Missing runner file: {RUNNER_PATH}")

if str(SCRIPTS_DIR) not in sys.path:
    sys.path.append(str(SCRIPTS_DIR))

import phase5_adaptive_wavelet_model as p5
importlib.reload(p5)

OUT_TABLES = ROOT / "results" / "tables" / "phase5"
OUT_PREDS = ROOT / "results" / "predictions" / "phase5_dbloss_full"
OUT_CKPTS = ROOT / "results" / "checkpoints" / "phase5_dbloss_full"

OUT_TABLES.mkdir(parents=True, exist_ok=True)
OUT_PREDS.mkdir(parents=True, exist_ok=True)
OUT_CKPTS.mkdir(parents=True, exist_ok=True)

MASTER_PATH = ROOT / "results" / "tables" / "master_all_models_metrics_unified.csv"
BEST_PATH = ROOT / "results" / "tables" / "master_best_by_dataset_unified.csv"

# --------------------------------------------
# 1) Hardware / efficiency setup
# --------------------------------------------
if torch.cuda.is_available():
    torch.backends.cudnn.benchmark = True
    if hasattr(torch, "set_float32_matmul_precision"):
        torch.set_float32_matmul_precision("high")
    GPU_NAME = torch.cuda.get_device_name(0)
    GPU_MEM_GB = torch.cuda.get_device_properties(0).total_memory / (1024**3)
else:
    GPU_NAME = "CPU"
    GPU_MEM_GB = 0.0

USE_AMP = bool(torch.cuda.is_available()) and bool(getattr(p5, "TRAIN_CFG", {}).get("use_amp", True))
SEED = 42
PATIENCE = int(getattr(p5, "TRAIN_CFG", {}).get("full_patience", 8))

print("Imported p5 from:", RUNNER_PATH)
print("DEVICE:", p5.DEVICE)
print("GPU_NAME:", GPU_NAME)
print("GPU_MEM_GB:", round(GPU_MEM_GB, 2))
print("USE_AMP:", USE_AMP)
print("PATIENCE:", PATIENCE)

# --------------------------------------------
# 2) Fixed architecture and DBLoss config
# --------------------------------------------
ALL_DATASETS = list(p5.ALL_DATASETS)

BEST_ARCH = {
    "levels": 2,
    "wavelet_family": "Db4",
    "d_model": 128,
    "num_backbone_blocks": 2,
    "dropout": 0.10,
    "filter_reg_lambda": 1e-4,
    "gate_entropy_lambda": 1e-4,
}

# Practical decomposition-aware loss inspired by DBLoss idea:
# supervise both trend and seasonal/residual structure on forecast horizon.
DBLOSS_CFG = {
    "lambda_decomp": 0.35,
    "trend_weight": 1.0,
    "seasonal_weight": 1.0,
    "kernel_map": {
        "etth1": 25,
        "etth2": 25,
        "ettm1": 49,
        "ettm2": 49,
        "weather": 49,
        "electricity": 25,
        "traffic": 25,
        "exchange": 7,
        "ili": 7,
    },
}

# GPU-aware batch multiplier:
# Safe bump on large-memory GPUs, but keep traffic/electricity conservative.
if GPU_MEM_GB >= 80:
    BATCH_MUL = {
        "etth1": 2,
        "etth2": 2,
        "ettm1": 2,
        "ettm2": 2,
        "weather": 2,
        "electricity": 1,
        "traffic": 1,
        "exchange": 2,
        "ili": 2,
    }
else:
    BATCH_MUL = {ds: 1 for ds in ALL_DATASETS}

print("ALL_DATASETS:", ALL_DATASETS)
print("BEST_ARCH:", BEST_ARCH)
print("DBLOSS_CFG:", DBLOSS_CFG)
print("BATCH_MUL:", BATCH_MUL)

# --------------------------------------------
# 3) Helpers
# --------------------------------------------
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

def mse(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=np.float64)
    y_pred = np.asarray(y_pred, dtype=np.float64)
    return float(np.mean((y_true - y_pred) ** 2))

def mae(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=np.float64)
    y_pred = np.asarray(y_pred, dtype=np.float64)
    return float(np.mean(np.abs(y_true - y_pred)))

def rmse(y_true, y_pred):
    return float(np.sqrt(mse(y_true, y_pred)))

def resolve_seq_len(ds):
    seq_candidates = p5.resolve_seq_candidates(ds, "searched")
    return int(seq_candidates[min(1, len(seq_candidates) - 1)])

def resolve_batch_size(ds):
    base = int(p5.DEFAULT_BATCH_SIZE[ds])
    return int(base * BATCH_MUL[ds])

def resolve_epochs(ds):
    return int(p5.DEFAULT_EPOCHS[ds])

def safe_kernel(kernel, pred_len):
    k = int(kernel)
    if pred_len <= 1:
        return 1
    if k >= pred_len:
        k = pred_len - 1 if pred_len % 2 == 0 else pred_len
    if k < 3:
        k = 3 if pred_len >= 3 else 1
    if k % 2 == 0:
        k -= 1
    if k < 1:
        k = 1
    return k

def moving_average_trend(x, kernel):
    # x: [B, H]
    if kernel <= 1:
        return x
    pad = kernel // 2
    x3 = x.unsqueeze(1)  # [B,1,H]
    xpad = F.pad(x3, (pad, pad), mode="replicate")
    trend = F.avg_pool1d(xpad, kernel_size=kernel, stride=1)
    return trend.squeeze(1)

def decomposition_aware_loss(pred_scaled, y_scaled, dataset_name, cfg):
    base = F.mse_loss(pred_scaled, y_scaled)

    kernel = safe_kernel(cfg["kernel_map"][dataset_name], pred_scaled.shape[1])

    trend_pred = moving_average_trend(pred_scaled, kernel)
    trend_true = moving_average_trend(y_scaled, kernel)

    seasonal_pred = pred_scaled - trend_pred
    seasonal_true = y_scaled - trend_true

    trend_loss = F.mse_loss(trend_pred, trend_true)
    seasonal_loss = F.mse_loss(seasonal_pred, seasonal_true)

    total = (
        base
        + cfg["lambda_decomp"] * (
            cfg["trend_weight"] * trend_loss
            + cfg["seasonal_weight"] * seasonal_loss
        )
    )

    parts = {
        "base_mse": float(base.detach().cpu().item()),
        "trend_loss": float(trend_loss.detach().cpu().item()),
        "seasonal_loss": float(seasonal_loss.detach().cpu().item()),
        "kernel": kernel,
    }
    return total, parts

@torch.no_grad()
def evaluate_model(model, loader, target_mean, target_std):
    model.eval()
    preds = []
    trues = []

    for batch in loader:
        x = batch["x_scaled"].to(p5.DEVICE, non_blocking=True)
        y_raw = batch["y_raw"].cpu().numpy()

        with torch.cuda.amp.autocast(enabled=USE_AMP):
            pred_scaled, aux = model(x)

        pred_raw = pred_scaled.detach().cpu().numpy() * target_std + target_mean
        preds.append(pred_raw)
        trues.append(y_raw)

    preds = np.concatenate(preds, axis=0)
    trues = np.concatenate(trues, axis=0)

    return {
        "preds": preds,
        "trues": trues,
        "mse": mse(trues.reshape(-1), preds.reshape(-1)),
        "mae": mae(trues.reshape(-1), preds.reshape(-1)),
        "rmse": rmse(trues.reshape(-1), preds.reshape(-1)),
    }

def metric_row_from_npz(dataset, model, family, seq_len, pred_len, pred_file_rel):
    p = ROOT / str(pred_file_rel)
    arr = np.load(p, allow_pickle=True)
    preds = np.asarray(arr["preds"], dtype=np.float64).reshape(-1)
    trues = np.asarray(arr["trues"], dtype=np.float64).reshape(-1)
    return {
        "dataset": dataset,
        "model": model,
        "family": family,
        "seq_len": int(seq_len),
        "pred_len": int(pred_len),
        "test_mse": mse(trues, preds),
        "test_mae": mae(trues, preds),
        "test_rmse": rmse(trues, preds),
        "prediction_file": str(pred_file_rel),
    }

# --------------------------------------------
# 4) One full dataset run
# --------------------------------------------
def train_one_dataset(dataset_name):
    set_seed(SEED)

    bundle = p5.load_bundle(dataset_name, input_mode="multivariate")
    pred_len = p5.resolve_pred_len(dataset_name, "long")
    seq_len = resolve_seq_len(dataset_name)
    batch_size = resolve_batch_size(dataset_name)
    epochs = resolve_epochs(dataset_name)

    train_loader, val_loader, test_loader = p5.make_loaders(bundle, seq_len, pred_len, batch_size)

    target_idx = bundle["target_idx"]
    target_mean = float(bundle["scaler_mean"][target_idx])
    target_std = float(bundle["scaler_std"][target_idx])

    model = p5.AdaptiveWaveletMixer(
        input_dim=int(bundle["values_scaled"].shape[1]),
        pred_len=pred_len,
        d_model=int(BEST_ARCH["d_model"]),
        levels=int(BEST_ARCH["levels"]),
        wavelet_family=BEST_ARCH["wavelet_family"],
        num_backbone_blocks=int(BEST_ARCH["num_backbone_blocks"]),
        dropout=float(BEST_ARCH["dropout"]),
        filter_reg_lambda=float(BEST_ARCH["filter_reg_lambda"]),
        gate_entropy_lambda=float(BEST_ARCH["gate_entropy_lambda"]),
    ).to(p5.DEVICE)

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=1e-3,
        weight_decay=1e-5,
        foreach=False,
    )

    scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP)

    best_val_rmse = float("inf")
    best_epoch = -1
    best_state = None
    patience_left = PATIENCE

    history = []
    t0 = time.time()

    for epoch in range(1, epochs + 1):
        model.train()
        train_losses = []
        train_base = []
        train_trend = []
        train_seasonal = []

        for batch in train_loader:
            x = batch["x_scaled"].to(p5.DEVICE, non_blocking=True)
            y = batch["y_scaled"].to(p5.DEVICE, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)

            with torch.cuda.amp.autocast(enabled=USE_AMP):
                pred_scaled, aux = model(x)
                loss, parts = decomposition_aware_loss(pred_scaled, y, dataset_name, DBLOSS_CFG)

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            train_losses.append(float(loss.detach().cpu().item()))
            train_base.append(parts["base_mse"])
            train_trend.append(parts["trend_loss"])
            train_seasonal.append(parts["seasonal_loss"])

        val_eval = evaluate_model(model, val_loader, target_mean, target_std)
        val_rmse = float(val_eval["rmse"])
        mean_train_loss = float(np.mean(train_losses)) if len(train_losses) > 0 else np.nan

        history.append({
            "dataset": dataset_name,
            "epoch": epoch,
            "train_loss": mean_train_loss,
            "train_base_mse": float(np.mean(train_base)) if len(train_base) > 0 else np.nan,
            "train_trend_loss": float(np.mean(train_trend)) if len(train_trend) > 0 else np.nan,
            "train_seasonal_loss": float(np.mean(train_seasonal)) if len(train_seasonal) > 0 else np.nan,
            "val_rmse": val_rmse,
            "val_mae": float(val_eval["mae"]),
            "val_mse": float(val_eval["mse"]),
            "decomp_kernel": safe_kernel(DBLOSS_CFG["kernel_map"][dataset_name], pred_len),
        })

        print(
            f"[{dataset_name}] epoch {epoch:03d} | "
            f"train_loss={mean_train_loss:.6f} | val_rmse={val_rmse:.6f} | "
            f"batch_size={batch_size}"
        )

        if val_rmse < best_val_rmse:
            best_val_rmse = val_rmse
            best_epoch = epoch
            best_state = copy.deepcopy(model.state_dict())
            patience_left = PATIENCE
        else:
            patience_left -= 1
            if patience_left <= 0:
                print(f"[{dataset_name}] early stopping at epoch {epoch}")
                break

    model.load_state_dict(best_state)

    test_eval = evaluate_model(model, test_loader, target_mean, target_std)

    ckpt_path = OUT_CKPTS / f"{dataset_name}_AdaptiveWaveletMixer_DBLoss.pt"
    pred_path = OUT_PREDS / f"{dataset_name}_AdaptiveWaveletMixer_DBLoss_test_predictions.npz"

    torch.save({
        "dataset": dataset_name,
        "model": "AdaptiveWaveletMixer_DBLoss",
        "best_arch": BEST_ARCH,
        "dbloss_cfg": DBLOSS_CFG,
        "seq_len": seq_len,
        "pred_len": pred_len,
        "best_epoch": best_epoch,
        "state_dict": model.state_dict(),
    }, ckpt_path)

    np.savez_compressed(
        pred_path,
        preds=test_eval["preds"],
        trues=test_eval["trues"],
        seq_len=seq_len,
        pred_len=pred_len,
    )

    row = {
        "dataset": dataset_name,
        "model": "AdaptiveWaveletMixer_DBLoss",
        "family": "wavelet_adaptive",
        "use_dbloss": True,
        "seq_len": seq_len,
        "pred_len": pred_len,
        "batch_size": batch_size,
        "epochs_target": epochs,
        "best_epoch": int(best_epoch),
        "train_seconds": float(time.time() - t0),
        "checkpoint": str(ckpt_path.relative_to(ROOT)),
        "prediction_file": str(pred_path.relative_to(ROOT)),
        "test_mse": float(test_eval["mse"]),
        "test_mae": float(test_eval["mae"]),
        "test_rmse": float(test_eval["rmse"]),
    }

    hist_df = pd.DataFrame(history)

    del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return row, hist_df

# --------------------------------------------
# 5) Run all 9 datasets
# --------------------------------------------
all_rows = []
all_hist = []

print("\nStarting full DBLoss run on all 9 datasets...\n")

for ds in ALL_DATASETS:
    print("\n" + "=" * 120)
    print(f"RUNNING FULL DBLOSS | dataset={ds}")
    print("=" * 120)
    row, hist_df = train_one_dataset(ds)
    all_rows.append(row)
    all_hist.append(hist_df)

full_df = pd.DataFrame(all_rows)
hist_df = pd.concat(all_hist, ignore_index=True)

metrics_csv = OUT_TABLES / "phase5_dbloss_full_metrics.csv"
history_csv = OUT_TABLES / "phase5_dbloss_full_history.csv"
registry_csv = OUT_TABLES / "phase5_dbloss_full_prediction_registry.csv"

full_df.to_csv(metrics_csv, index=False)
hist_df.to_csv(history_csv, index=False)
full_df[[
    "dataset", "model", "family", "seq_len", "pred_len",
    "checkpoint", "prediction_file", "test_mse", "test_mae", "test_rmse"
]].to_csv(registry_csv, index=False)

print("\nSaved full metrics:", metrics_csv)
print("Saved history:", history_csv)
print("Saved prediction registry:", registry_csv)
display(full_df)

# --------------------------------------------
# 6) Build merged candidate master tables
# --------------------------------------------
if not MASTER_PATH.exists():
    raise FileNotFoundError(f"Missing master file: {MASTER_PATH}")
if not BEST_PATH.exists():
    raise FileNotFoundError(f"Missing best file: {BEST_PATH}")

master = pd.read_csv(MASTER_PATH)
best_old = pd.read_csv(BEST_PATH)

# Remove older rows of same model if any
master_candidate = master[master["model"] != "AdaptiveWaveletMixer_DBLoss"].copy()

new_rows = []
for _, r in full_df.iterrows():
    new_rows.append(
        metric_row_from_npz(
            dataset=r["dataset"],
            model=r["model"],
            family=r["family"],
            seq_len=r["seq_len"],
            pred_len=r["pred_len"],
            pred_file_rel=r["prediction_file"],
        )
    )

new_df = pd.DataFrame(new_rows)
master_candidate = pd.concat([master_candidate, new_df], ignore_index=True)
master_candidate = master_candidate.sort_values(["dataset", "family", "model"]).reset_index(drop=True)

best_candidate = (
    master_candidate.sort_values(["dataset", "test_rmse", "test_mae", "test_mse"])
                    .groupby("dataset", as_index=False)
                    .first()[["dataset", "model", "family", "test_mse", "test_mae", "test_rmse"]]
)

candidate_master_csv = ROOT / "results" / "tables" / "master_all_models_metrics_unified_dbloss_candidate.csv"
candidate_best_csv = ROOT / "results" / "tables" / "master_best_by_dataset_unified_dbloss_candidate.csv"

master_candidate.to_csv(candidate_master_csv, index=False)
best_candidate.to_csv(candidate_best_csv, index=False)

print("\nSaved candidate merged master:", candidate_master_csv)
print("Saved candidate best table:", candidate_best_csv)

# --------------------------------------------
# 7) Compare DBLoss against current best
# --------------------------------------------
compare = best_old.rename(columns={
    "model": "old_best_model",
    "family": "old_best_family",
    "test_mse": "old_best_mse",
    "test_mae": "old_best_mae",
    "test_rmse": "old_best_rmse",
}).merge(
    full_df[["dataset", "model", "test_mse", "test_mae", "test_rmse"]].rename(columns={
        "model": "dbloss_model",
        "test_mse": "dbloss_mse",
        "test_mae": "dbloss_mae",
        "test_rmse": "dbloss_rmse",
    }),
    on="dataset",
    how="left"
)

compare["rmse_gain_abs"] = compare["old_best_rmse"] - compare["dbloss_rmse"]
compare["rmse_gain_pct"] = 100.0 * compare["rmse_gain_abs"] / compare["old_best_rmse"]

compare["mae_gain_abs"] = compare["old_best_mae"] - compare["dbloss_mae"]
compare["mae_gain_pct"] = 100.0 * compare["mae_gain_abs"] / compare["old_best_mae"]

compare_csv = OUT_TABLES / "phase5_dbloss_full_vs_current_best.csv"
compare.to_csv(compare_csv, index=False)

wins = int((compare["rmse_gain_abs"] > 0).sum())
losses = int((compare["rmse_gain_abs"] < 0).sum())
ties = int((compare["rmse_gain_abs"] == 0).sum())

print("\nSaved comparison table:", compare_csv)
display(compare)

print("\n" + "=" * 120)
print("FULL DBLOSS DECISION")
print("=" * 120)
print(f"DBLoss full run wins on {wins}/{len(ALL_DATASETS)} datasets by RMSE")
print(f"DBLoss full run loses on {losses}/{len(ALL_DATASETS)} datasets by RMSE")
print(f"Ties: {ties}")

if wins >= 4:
    print("RECOMMENDATION: DBLoss full run is promising enough to keep as the main improved candidate.")
else:
    print("RECOMMENDATION: DBLoss full run is not strong enough overall. Then the next move should be adaptive lookback, not more search.")

print("\nCandidate best-by-dataset table:")
display(best_candidate)